## How do sentiments towards COVID-19 vaccine compare before and after the emergence of the Omicron variant in the Arab world?
### A Twitter comparative sentiment analysis of the pre-Omicron and post-Omicron phases by Arabic users.

### Dependency - The analysis leverages CAMeL Tools
### CAMeL Tools is a suite of Arabic natural language processing tools developed by the CAMeL Lab at New York University Abu Dhabi.
#### pip3 install camel_tools

In [ ]:
from camel_tools.utils.normalize          import normalize_alef_maksura_ar
from camel_tools.utils.normalize          import normalize_alef_ar
from camel_tools.utils.normalize          import normalize_teh_marbuta_ar
from camel_tools.utils.normalize          import normalize_unicode
from camel_tools.dialectid                import DialectIdentifier
from camel_tools.sentiment                import SentimentAnalyzer
from camel_tools.utils.dediac             import dediac_ar
from camel_tools.morphology.database      import MorphologyDB
from camel_tools.morphology.analyzer      import Analyzer
from camel_tools.tokenizers.word          import simple_word_tokenize
from camel_tools.disambig.mle             import MLEDisambiguator
from camel_tools.tokenizers.morphological import MorphologicalTokenizer
import os

os.environ["CAMELTOOLS_DATA"] = "~/.camel_tools"


text = 'ﷺ'

sentence = "sentence from tweet"

sent_norm = normalize_unicode(sentence)


# Normalize alef variants to 'ا'
sent_norm = normalize_alef_ar(sentence)

# Normalize alef maksura 'ى' to yeh 'ي'
sent_norm = normalize_alef_maksura_ar(sent_norm)

# Normalize teh marbuta 'ة' to heh 'ه'
sent_norm = normalize_teh_marbuta_ar(sent_norm)


## Import raw data
### Please see Data_by_day notebook for the method by which the pre and post files were created
#### Tweets using the keywords “لقاحات” or “تطعيم” or “لقاح” or “تطعيمات”  - ”vaccines”, “inoculation”, ”vaccine”, “inoculations” 

### Sentiment Analysis

In [ ]:
# Load the morphological database.
# The MorphologyDB database is used for analyzing modern Standard Arabic. 
db = MorphologyDB.builtin_db()

analyzer = Analyzer(db)

analyses = analyzer.analyze('موظف')

In [ ]:
from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.tokenizers.morphological import MorphologicalTokenizer
import arabicstopwords.arabicstopwords as stp
# The tokenizer expects pre-tokenized text
sentence = simple_word_tokenize("sentence from tweet")

# Load a pretrained disambiguator to use with a tokenizer
mle = MLEDisambiguator.pretrained('calima-msa-r13')

# Without providing additional arguments, the tokenizer will output undiacritized
# morphological tokens for each input word delimited by an underscore.
tokenizer = MorphologicalTokenizer(mle, scheme='d3tok')
tokens = tokenizer.tokenize(sentence)

# By specifying `split=True`, the morphological tokens are output as seperate
# strings.
tokenizer = MorphologicalTokenizer(mle, scheme='d3tok', split=True)
tokens = tokenizer.tokenize(sentence)

# We can output diacritized tokens by setting `diac=True`
tokenizer = MorphologicalTokenizer(mle, scheme='d3tok', split=True, diac=True)
tokens = tokenizer.tokenize(sentence)

### Plot sentiments Pre and Post Omicron

In [ ]:
import pandas as pd
#Read pre-Omicron tweets from the csv file into a dataframe
raw_tweets_pre = pd.read_csv('tweet_data_by_day.csv', index_col=0)
raw_tweets_pre = raw_tweets_pre[ raw_tweets_pre['text'].str.startswith('RT') == False ]

#Read post-Omicron tweets from the csv file into a dataframe
raw_tweets_post = pd.read_csv('tweet_data_by_day_post_omicron.csv', index_col=0)
raw_tweets_post = raw_tweets_post[ raw_tweets_post['text'].str.startswith('RT') == False ]

In [ ]:
#(optional) use time to see how long it takes to compute sentiments and dialect in your data
from datetime import datetime
#apply dialect identifier and sentiment analyzer then merge them together into one dataframe
did = DialectIdentifier.pretrained()
sa = SentimentAnalyzer.pretrained()

t1 = datetime.now()
#for pre-Omicron tweets
raw_tweets_pre[ 'Sentiment' ] = raw_tweets_pre['text'].apply( lambda x: sa.predict(  [ x ] )[ 0 ] )
raw_tweets_pre[ 'Label' ]     = raw_tweets_pre['text'].apply( lambda x: did.predict( [ x ] )[ 0 ].top )

# Read the CAMeL Tools Dialect Identifier locations document saved on your computer (locations contains labels for cities identified by CAMeL Tools dialect identifier )
loc = pd.read_csv( '/your_directory/locations.csv' );  
raw_tweets_pre = raw_tweets_pre.merge( loc, on ='Label' )
#write the dataframe into a csv file for later use
raw_tweets_pre.to_csv('Raw_tweets_plus_sentiments_pre.csv', index = False )

print(datetime.now() - t1)


In [ ]:
#Repeat for post_Omicron
t1 = datetime.now()

raw_tweets_post[ 'Sentiment' ] = raw_tweets_post['text'].apply( lambda x: sa.predict(  [ x ] )[ 0 ] )
raw_tweets_post[ 'Label' ]     = raw_tweets_post['text'].apply( lambda x: did.predict( [ x ] )[ 0 ].top )

# Read the CAMeL Tools Dialect Identifier label document saved on your computer location
loc = pd.read_csv( '/your_directory/locations.csv' );  
raw_tweets_post = raw_tweets_post.merge( loc, on ='Label' )
#write the dataframe into a csv for later use
raw_tweets_post.to_csv('Raw_tweets_plus_sentiments_post.csv', index = False )

print(datetime.now() - t1)

In [ ]:
#Read the saved csv files into a dataframe
df1_tweets = pd.read_csv('Raw_tweets_plus_sentiments_pre.csv', low_memory=False);
df2_tweets = pd.read_csv('Raw_tweets_plus_sentiments_post.csv', low_memory=False)

#Apply any modification for your file, for ex: replaced a misspellled city with correct spelling, and shortened a label "MSA"
df1_tweets.City.replace( to_replace = ("Jeddha", "Modern Standard Arabic"), value=("Jeddah","MSA"), inplace=True)
df2_tweets.City.replace( to_replace = ("Jeddha", "Modern Standard Arabic"), value=("Jeddah","MSA"), inplace=True)
#If replacing 2 values create an error, replace one at a time like in 
#df1_tweets.City.replace( to_replace = "Modern Standard Arabic", value="MSA", inplace=True)

#Adjust date format as year month only for later use
df1_tweets.created_at = df1_tweets.created_at.apply(lambda x: x[:10] )



In [ ]:
#create a matrix of sentiments by city
import matplotlib.pyplot as plt
from collections import Counter
import pprint as pp
titles = ['sentiment', 'city', 'country', 'region'] 
sentiment_type = ['positive', 'negative', 'neutral'] 

def build_matrix(sentiments):
    matrix = {}

    for row in sentiments.values.astype(str).tolist():
        if not matrix.get(row[0]):
            matrix[ row[0] ] = {}

        if not matrix[ row[0] ].get( row[1] ) :
            matrix[ row[0] ] [ row[1] ] = {}
            matrix[ row[0] ] [ row[1] ] = 1      
        else:
            matrix[ row[0] ] [ row[1] ] += 1      
            
    return(matrix)

In [ ]:
#Optional plots. I ended up not using them. Instead I plotted the 2 data frames in one plot for comparison 
FIGS_PATH='.'

df1 = pd.DataFrame( build_matrix( df1_tweets[['Sentiment','City']] ))

#plot sentiments distriution for pre-Omicron tweets
sentiment_counts = (df1['positive'].sum(), df1['negative'].sum(), df1['neutral'].sum())
labels=["Positive","Negative", "Neutral"]
fig1, ax1 = plt.subplots(figsize=(8, 8))
ax1.pie(sentiment_counts, labels=labels, autopct='%1.1f%%', shadow=False, startangle=90 )
ax1.axis('equal')  
plt.tight_layout()
plt.title('Sentiment Distribution for Pre-Omicron Dataset')
plt.savefig("sentiment1.png")
plt.show()


In [ ]:
#Optional plot
#FIGS_PATH='Insert_Your_Fig_Path'

FIGS_PATH='.'

# Creating a data frame using pandas to compare the occurrence of each sentiment by city for the pre_Omicron corpus
from datetime import datetime

df1 = pd.DataFrame( build_matrix( df1_tweets[['Sentiment','City']] ))

t = datetime.now()
ax= df1.sort_index().plot.bar(color = ('#FF8C00','#228B22', '#1E90FF'), figsize=(20, 10))
ax.set_xlabel('Cities', fontsize=20)
ax.set_ylabel('Frequency', fontsize=20)
plt.xticks(fontsize=12)
plt.legend(fontsize = 20)
#plt.savefig(FIGS_PATH+"/sentiments_by_city_pre.png")
plt.show()

print(datetime.now() - t)


### Processing

In [12]:
#add Arabic stopwords from this file
with open('ar_stopwords.txt', 'r') as file:
    stopwords_from_file = file.read()

In [13]:
from camel_tools.utils.normalize import normalize_unicode
#remove stopwords and all characters that are not arabic letters or # numbers and lemmatize the words
def preprocess_ar(text):
    processedText = []
    
    my_stp = stp.stopwords_list() + [ u'كورونا', u'ان' ]

    # Create Lemmatizer and Stemmer.
    st = ISRIStemmer()

    for t in text:
        t = ''.join(c for c in t if ud.category(c) == 'Lo' or ud.category(c) == 'Nd' or c == ' ')
   
        commentwords = ''
        for word in t.split():
            # Checking if the word is a stopword.

            if word not in my_stp:
                if len(word)>1:
                    # Lemmatizing the word.
                    word = st.suf32(word)
                    commentwords += (word+' ')
        processedText.append(normalize_unicode(commentwords))
    
    return processedText  

In [14]:
from nltk.stem.isri import ISRIStemmer
import unicodedata as ud
import arabicstopwords.arabicstopwords as stp

processedtext_ar1 = preprocess_ar( df1_tweets.text )
processedtext_ar2 = preprocess_ar( df2_tweets.text )


### Analysis

In [15]:
#Some imports were already imported above, but repeated here to note that they are needed in this section
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ar_wordcloud import ArabicWordCloud
import nltk

### Visualization
#### Word Cloud

In [ ]:
#Add Arabic stopwords to the natural language toolkit
AddStopwords = ["additional_list_of_words_to_exclude_goes_here e.g. u'لقاح' "]
stopwords = nltk.corpus.stopwords.words('arabic')
stopwords.extend(['Arabic stopwords'])
stopwords.append( stopwords_from_file)

#plot Arabic wordcloud
awc = ArabicWordCloud(width=2000,height=1600,max_font_size=400,max_words=10000,collocations=False, background_color='skyblue', colormap="Purples")
plt.figure(figsize=(16,16))
#save figure
wc_ar = awc.from_text(u''.join(processedtext_ar1))
plt.axis("off")
plt.imshow(wc_ar)
plt.savefig("WC1.png")



In [ ]:
# Create a WordCloud object for translated words for pre_Omicron
#res2_post=  { translator.translate(k): v for k,v in res_post.items() }
my_dict2= {"your_dictionary_here" }
df = pd.DataFrame(my_dict2.items(), columns=['word', 'count'])
wordcloud = WordCloud(
    collocations=False, background_color='skyblue', colormap="Purples",  # Set background color
    width=800,  # Set width of the cloud image
    height=400,  # Set height of the cloud image
    max_words=100,  # Set the maximum number of words to display
).generate(' '.join(my_dict2.keys()))  # Generate the word cloud

# Display the word cloud
plt.figure(figsize=(8, 8), facecolor=None)
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.tight_layout(pad=0)
plt.savefig("WC2E.png")
plt.show()

### Frequencies

In [ ]:
#count the most common words found in tweets 
#Build a dictionary for pre_Omicron (repeat for post-Omicron)
counts_pre = dict()
for i in range(0,len(processedtext_ar1)):
    for word in processedtext_ar1[i].split(' '):
        if  not counts_pre.get(''.join(word)):
            counts_pre[ u''.join(word) ] = 1
        else:
            counts_pre[ u''.join(word) ] += 1
            

In [ ]:
#plot the most common words found in tweets for pre_Omicron (repeat for post-Omicron)
import seaborn as sns
import pandas as pd
import arabic_reshaper
from bidi.algorithm import get_display
import matplotlib.pyplot as plt
#use Arabic reshaper to display Arabic letters correctly from right to left
res_pre=  dict(sorted(counts_pre.items(), key=lambda item: item[1], reverse=True)[:20])
del res_pre['']
print(res_pre)

res2_pre = { get_display(arabic_reshaper.reshape(k)): v for k,v in res_pre.items() }
#put the words in a data frame
df = pd.DataFrame(res2_pre.items(), columns=['word', 'count'])
#plot frequencies
sns.set(style="whitegrid")
sns.set_theme(rc={'figure.figsize':(11.7,8.27)})
ax = sns.barplot(x="count", y="word", data=df)


In [ ]:
#Most Common Words translated to English for pre-Omicon tweets
# Translated words are the output of the counts above

my_dict1= {arabic_reshaper.reshape('لقاح')[::-1] + ' / ' + 'vaccine' :17133, 
           arabic_reshaper.reshape('تلقيح')[::-1] + ' / ' +'inoculation': 3217, 
           arabic_reshaper.reshape('حضور')[::-1] + ' / ' +'presence':0, 
           arabic_reshaper.reshape('جرعة')[::-1] + ' / ' +'dose': 3298,   
           arabic_reshaper.reshape('ينزل')[::-1] + ' / ' +'drop': 0, 
           arabic_reshaper.reshape('حضورالجرعة')[::-1] + ' / ' +'dose availability':0, 
           arabic_reshaper.reshape('تطعيم')[::-1] + ' / ' +'vaccination': 666, 
           arabic_reshaper.reshape('انجاز')[::-1] + ' / ' +'accomplishment':0, 
           arabic_reshaper.reshape('تاخذ')[::-1] + ' / ' +'uptake':0, 
           arabic_reshaper.reshape('الثانيه')[::-1] + ' / ' +'the second':0, 
           arabic_reshaper.reshape('تصفير')[::-1] + ' / ' +'zeroing':0, 
           arabic_reshaper.reshape('هروب')[::-1] + ' / ' +'escape':0, 
           arabic_reshaper.reshape('اولى')[::-1] + ' / ' +'first': 0,  
           arabic_reshaper.reshape('جرعة ثانية')[::-1] + ' / ' +'second dose':0,  
           arabic_reshaper.reshape('الله')[::-1] + ' / ' +'God':1520,  
           arabic_reshaper.reshape('فايزر')[::-1] + ' / ' +'Pfizer': 1389, 
           arabic_reshaper.reshape('الصحة')[::-1] + ' / ' +'the health': 1168, 
           arabic_reshaper.reshape('جرعات')[::-1] + ' / ' +'doses': 1106, 
           arabic_reshaper.reshape('محصن')[::-1] + ' / ' +'protected': 935,   
           arabic_reshaper.reshape('اخذ')[::-1] + ' / ' +'took': 864 , 
           arabic_reshaper.reshape('السلام')[::-1] + ' / ' +'peace': 802, 
           arabic_reshaper.reshape('موعد')[::-1] + ' / ' +'appointment': 719, 
           arabic_reshaper.reshape('توكل')[::-1] + ' / ' +'surrendering': 689,   
           arabic_reshaper.reshape('الغاء')[::-1] + ' / ' +'cancellation': 587   }
df = pd.DataFrame(my_dict1.items(), columns=['word', 'count'])

sns.set(style="whitegrid")
sns.set_theme(rc={'figure.figsize':(11.7,8.27)})
ax = sns.barplot(x="count", y="word", data=df)

In [ ]:
#Plot the dictionaries in one graph after normalizing the data using the MinMax method
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# my_dict1 = { keys and values of the translated most frequent words }
#same dor my_dict2
## Plot the 2 dictionaries side by side

dfdict1 = pd.DataFrame( columns=['Words', 'Values'] )
dfdict2 = pd.DataFrame( columns=['Words', 'Values'] )

dfdict1.Words=my_dict1.keys()
dfdict1.Values=my_dict1.values()

dfdict2.Words=my_dict2.keys()
dfdict2.Values=my_dict2.values()

dfdict1['Norm'] = (dfdict1['Values'] - np.min(dfdict1['Values'])) / (np.max(dfdict1['Values']) - np.min(dfdict1['Values']))
dfdict2['Norm'] = (dfdict2['Values'] - np.min(dfdict2['Values'])) / (np.max(dfdict2['Values']) - np.min(dfdict2['Values']))

bar_width = .35
# Set the x locations for the bars
x1 = range(len(dfdict1))
x2 = [x + bar_width for x in x1]

plt.bar(x1, dfdict1['Norm'], width=bar_width, label='Pre-Omicron', color='green')
plt.bar(x2, dfdict2['Norm'], width=bar_width, label='Post-Omicron', color='orange')

# Add labels and title
#plt.xlabel()
plt.ylabel('Normalized Values', fontsize=15)
plt.title('Most Frequent Words', fontsize=15)
plt.xticks([r + bar_width / 2 for r in range(len(dfdict1))], dfdict1['Words'], rotation=80, fontsize=12)
plt.legend()
plt.figure(figsize=(4, 3))
plt.savefig("Most Frequent Words_Normalized.png",)
#plt.savefig("Most Frequent Words_Normalized.png", facecolor='yellow', bbox_inches="tight", pad_inches=0.3, transparent=False)
# Show the plot
plt.show()

In [27]:
#Create a new column in the dataframe to designate each phase of the data
df1_tweets['phase'] = 'pre_Omicron'
df2_tweets['phase'] = 'post_Omicron'
#Combine the dataframes on the columns you want to plot next
df_all = pd.concat([df1_tweets[['phase', 'Sentiment']], df2_tweets[['phase','Sentiment']]])
#Choose the columns to include for plotting the data later
df_all = pd.DataFrame( build_matrix( df_all[['Sentiment','phase']] ))
sentiment_counts = ( df_all['negative'].sum(), df_all['neutral'].sum(), df_all['positive'].sum())
labels=["Negative", "Neutral", "Positive"]

In [ ]:
#Normalize the data for adequate comparison 
#Calculate percentages of sentiments in each phase

all_pre = df_all['negative']['pre_Omicron'] + df_all['neutral']['pre_Omicron'] + df_all['positive']['pre_Omicron'] 
all_post = df_all['negative']['post_Omicron'] + df_all['neutral']['post_Omicron'] + df_all['positive']['post_Omicron'] 
dat=pd.DataFrame(columns=['negative', 'neutral', 'positive'], index=[ 'pre_Omicron', 'post_Omicron'] )

dat['negative']['pre_Omicron'] = df_all['negative']['pre_Omicron'] / all_pre * 100
dat['neutral']['pre_Omicron'] = df_all['neutral']['pre_Omicron'] / all_pre * 100
dat['positive']['pre_Omicron'] = df_all['positive']['pre_Omicron'] / all_pre * 100

dat['negative']['post_Omicron'] = df_all['negative']['post_Omicron'] / all_post * 100
dat['neutral']['post_Omicron'] = df_all['neutral']['post_Omicron'] / all_post * 100
dat['positive']['post_Omicron'] = df_all['positive']['post_Omicron'] / all_post * 100

dat.columns.values.tolist()
dat

In [ ]:
#Plot the percent sentiments side by side

ax = dat.plot(kind="bar", color =('#FF8C00', '#228B22', '#1E90FF' ), stacked=True, figsize=(7, 5), rot=0 )
plt.xticks(fontsize=12, rotation=0)
plt.yticks(fontsize=12)
for container in ax.containers:
    ax.bar_label(container,  label_type="center", color="black", fontsize=7, fmt='%.1f%%')
plt.title("Change in Sentiments in Twitter Data", fontsize=15)  
#plt.savefig("Change in Sentiments in Twitter Data")


In [ ]:
#Combine the dataframes so we can plot them together
df_combined=df1.join(df2)
df_combined

In [ ]:
# This step is necessary for normalizing the data. Use of the bar graph is optional

import numpy as np
from sklearn.preprocessing import MinMaxScaler
#Normalize the data using scaler
# # Initialize the scaler
scaler = MinMaxScaler()

# # Fit and transform the data using Min-Max Scaling

df1 = pd.DataFrame( build_matrix( df1_tweets[['Sentiment','City']] ))
print(np.max (df1.negative) )
df1.negative = ( df1.negative - np.min(df1.negative) ) / (np.max(df1.negative) - np.min(df1.negative))
df1.neutral = ( df1.neutral - np.min(df1.neutral) ) / (np.max(df1.neutral) - np.min(df1.neutral))
df1.positive = ( df1.positive - np.min(df1.positive) ) / (np.max(df1.positive) - np.min(df1.positive))
df1.rename(columns={'negative':'Negtive_pre_Omicron', 'positive' : 'Positive_pre_Omicron', 'neutral' : 'Neutral_pre_Omicron' }, inplace=True)

FIGS_PATH='.'

# Creating a data frame using pandas to compare the occurrence of each sentiment by city for the X corpus
from datetime import datetime
## Optional graph
t = datetime.now()
ax= df1.sort_index().plot.bar(color = ('#1E90FF','#228B22', '#FF8C00'), figsize=(20, 10))
ax.set_xlabel('Cities', fontsize=20)
ax.set_ylabel('Frequency', fontsize=20)
plt.xticks(fontsize=12)
plt.legend(fontsize = 20)
plt.savefig(FIGS_PATH+"/sentiments_by_city_pre.png")
plt.show()

print(datetime.now() - t)


In [ ]:
#Plot the normalized sentiments by city for the combined data
# Set up figure and axes
fig, ax = plt.subplots(figsize=(10, 6))
Colors = [ '#FF8C00','#228B22', '#FF7A5C',  '#1F5D3C','#C9633F', '#B4E01C'  ]
# Plot both bar graphs
df_combined.sort_index(axis=1)
df_combined.plot(kind="bar", stacked = True, color=Colors )
plt.xlabel('Cities', fontsize=14)
plt.ylabel('Sentiments normalized')
plt.title("Change in Sentiments by City in Twitter Data", fontsize=14)

plt.savefig("Change in Sentiments by City in Twitter Data")
